# BioPAKE: Biometric Password-Authenticated Key Exchange

A privacy-preserving biometric authentication protocol combining:
- **Cosine LSH** for biometric template protection
- **Pedersen commitments** and **zero-knowledge proofs** (Bulletproofs)
- **Oblivious PRF** for secure key derivation
- **Disjoint LSH bags** for tolerance to biometric variation

This notebook benchmarks the full protocol and evaluates false-positive / false-negative rates.


## 1. Setup

In [ ]:
import os, sys, time, csv, json, random, hashlib, secrets, statistics, ctypes, ctypes.util
from typing import List, Tuple
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd

from bsp import CosineLSH

# ── Precomputescalmult (compiled C extension) ─────────────────────────────────
sys.path.insert(0, os.path.join(os.getcwd(), 'precomputescalmult'))
from precomp_scalarmult import (
    PrecompPoint, RawPoint, batch_pedersen_commit, point_add_raw, point_sub_raw,
    scalarmult_rawpoint, batch_multiscalar_mult_raw, batch_pedersen_commit_raw,
    set_num_threads, get_num_threads, decode_to_raw, batch_short_msm_raw,
    batch_short_msm_auto_raw, batch_offset_subtract_raw, batch_pow2_mult_raw,
    batch_scalarmult_pairs_raw, batch_multiscalar_mult,
    batch_hash_enc, batch_sha256_tag_key,
    batch_secretbox_open_verify, secretbox_open,
    batch_nizk_hash_verify, batch_nizk_hash_prove,
)

# ── python_bulletproofs (Rust/PyO3 extension — run `maturin develop` to build) ─
import python_bulletproofs
from python_bulletproofs import (
    prove_range_aggregate_parallel, verify_range_aggregate_parallel,
)

# ── AES / KDF ────────────────────────────────────────────────────────────────
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad, unpad
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.backends import default_backend


### 1.1 Libsodium Bindings and EC Primitives

In [ ]:
SCALAR_LEN = 32
POINT_LEN = 32

def load_sodium():
    for name in ("sodium", "libsodium"):
        path = ctypes.util.find_library(name)
        if path:
            try: return ctypes.CDLL(path)
            except: pass
    conda_prefix = os.environ.get("CONDA_PREFIX")
    if conda_prefix:
        candidates = [
            os.path.join(conda_prefix, "Library", "bin", "libsodium.dll"),
            os.path.join(conda_prefix, "lib", "libsodium.so"),
        ]
        for c in candidates:
            if os.path.exists(c): return ctypes.CDLL(c)
    raise OSError("libsodium not found")

sodium = load_sodium()
if hasattr(sodium, "sodium_init"):
    sodium.sodium_init()

# --- Bindings ---

# Scalar Math
sodium.crypto_core_ristretto255_scalar_random.argtypes = [ctypes.c_void_p]
try:
    sodium.crypto_core_ristretto255_scalar_mul.argtypes = [ctypes.c_void_p, ctypes.c_void_p, ctypes.c_void_p]
    sodium.crypto_core_ristretto255_scalar_mul.restype = ctypes.c_int
except AttributeError:
    raise RuntimeError("libsodium version too old (missing scalar_mul).")
sodium.crypto_core_ristretto255_scalar_invert.argtypes = [ctypes.c_void_p, ctypes.c_void_p]

# Point Math
sodium.crypto_scalarmult_ristretto255_base.argtypes = [ctypes.c_void_p, ctypes.c_void_p] # P = n * G
sodium.crypto_scalarmult_ristretto255.argtypes = [ctypes.c_void_p, ctypes.c_void_p, ctypes.c_void_p] # P = n * Q

# NEW: Point Addition and Subtraction
try:
    # R = P + Q
    sodium.crypto_core_ristretto255_add.argtypes = [ctypes.c_void_p, ctypes.c_void_p, ctypes.c_void_p]
    sodium.crypto_core_ristretto255_add.restype = ctypes.c_int
    
    # R = P - Q
    sodium.crypto_core_ristretto255_sub.argtypes = [ctypes.c_void_p, ctypes.c_void_p, ctypes.c_void_p]
    sodium.crypto_core_ristretto255_sub.restype = ctypes.c_int
except AttributeError:
    raise RuntimeError("libsodium version too old (missing point add/sub).")


# --- Wrappers ---

def random_scalar() -> bytes:
    buf = (ctypes.c_ubyte * SCALAR_LEN)()
    sodium.crypto_core_ristretto255_scalar_random(ctypes.byref(buf))
    return bytes(buf)

def random_point() -> bytes:
    # To get a random valid point, we generate a random scalar and multiply by base
    return scalar_to_point(random_scalar())

def scalar_mul(x: bytes, y: bytes) -> bytes:
    z = (ctypes.c_ubyte * SCALAR_LEN)()
    sodium.crypto_core_ristretto255_scalar_mul(ctypes.byref(z), x, y)
    return bytes(z)

def scalar_invert(s: bytes) -> bytes:
    inv = (ctypes.c_ubyte * SCALAR_LEN)()
    sodium.crypto_core_ristretto255_scalar_invert(ctypes.byref(inv), s)
    return bytes(inv)

def scalar_to_point(s: bytes) -> bytes:
    p = (ctypes.c_ubyte * POINT_LEN)()
    sodium.crypto_scalarmult_ristretto255_base(ctypes.byref(p), s)
    return bytes(p)

def point_mul(scalar: bytes, point: bytes) -> bytes:
    out = (ctypes.c_ubyte * POINT_LEN)()
    sodium.crypto_scalarmult_ristretto255(ctypes.byref(out), scalar, point)
    return bytes(out)

def point_add(p: bytes, q: bytes) -> bytes:
    r = (ctypes.c_ubyte * POINT_LEN)()
    sodium.crypto_core_ristretto255_add(ctypes.byref(r), p, q)
    return bytes(r)

def point_sub(p: bytes, q: bytes) -> bytes:
    r = (ctypes.c_ubyte * POINT_LEN)()
    sodium.crypto_core_ristretto255_sub(ctypes.byref(r), p, q)
    return bytes(r)

def xor_bytes(a: bytes, b: bytes) -> bytes:
    return bytes(x ^ y for x, y in zip(a, b))


def scalar_negate(s: bytes) -> bytes:
    """
    Computes the mathematical negation of a scalar modulo the curve order.
    Returns -s mod q.
    """
    neg = (ctypes.c_ubyte * SCALAR_LEN)()
    sodium.crypto_core_ristretto255_scalar_negate(ctypes.byref(neg), s)
    return bytes(neg)


"""
NOTE: this is super important:
    Since we are adding a paderson commitment now, we need to ensure 

"""

L = (1 << 252) + 27742317777372353535851937790883648493

DALEK_H = python_bulletproofs.get_dalek_default_h()

DALEK_G = python_bulletproofs.get_dalek_default_g()

SCALAR_ONE = b'\x01' + b'\x00' * 31

def int_to_scalar_bytes(val: int) -> bytes:
    """Wraps integers safely around the Curve25519 order and serializes to 32 bytes."""
    L = (1 << 252) + 27742317777372353535851937790883648493
    scalar_int = val % L
    return scalar_int.to_bytes(32, byteorder='little')


# this is important for OPRF's HASH inside!!!!
def hash_point_to_scalar(point_bytes: bytes) -> int:
    h = hashlib.sha256(point_bytes).digest()
    return int.from_bytes(h, 'little') % L

## 2. Protocol Classes

### 2.1 FaceAuthenticationClient

Manages client-side ZK proofs over committed facial embeddings.

In [ ]:
class FaceAuthenticationClient:
    def __init__(self, facial_vector: np.ndarray, hypervectors: np.ndarray):
        self.facial_vector = facial_vector
        self.hypervectors = hypervectors
        
        self.data_bits = 12
        self.proof_bits = 16
        self.M = (1 << (self.data_bits - 1)) - 1  # 2047 shift for 12-bit
        

#---------------------------------------Saved Computation Results -----------------------------------------------------
        self.face_commitments = []
        self.face_blinding_factors = [] # NEW: We must track the initial blinders!
        self.face_proofs = []

        self.face_r_commitments = []  # R_i = r_i * G
        self.nizk_proofs = b""        # NIZK PoK for (C_i, R_i) — bytes(32 + n*64)

        self.dot_commitments = []
        self.dot_blinding_factors = [] # NEW: Tracking the homomorphic blinders

        self.bit_commitments = []
        self.bit_blinding_factors = []

        self.linkage_commitments = []
        self.linkage_proofs = []

        self.oprf_output= None # For initialization to be None 

        self.ptG = PrecompPoint(DALEK_G,expected_calls=10000)
        self.ptH = PrecompPoint(DALEK_H,expected_calls=10000)

#---------------------------------------Saved Computation Results (end) -----------------------------------------------------

        # Pre-compute plaintext LSH bits for the OPRF phase later
        self.raw_dot_products = np.dot(self.hypervectors, self.facial_vector)

        # 0 if Positive, 1 if Negative (Triggers the shift!)
        self.hashed_bits = [0 if dot >= 0 else 1 for dot in self.raw_dot_products]
    

    def CommitFace(self) -> Tuple[List[bytes], List[bytes], List, List[bytes], List]:
        """
        Step 1: Locks the shifted facial vector inside standard Pedersen Commitments (Base G)
        using Python-controlled random blinders so we can track them.
        Also produces R_i = r_i * G and a NIZK PoK proving (C_i, R_i) share the same r_i.
        """
        shiftlist = []
        for i, val in enumerate(self.facial_vector):
            shifted_val = int(val) + self.M
            shiftlist.append(shifted_val)
            if shifted_val < 0 or shifted_val >= (1 << self.data_bits):
                raise ValueError(f"Fatal: Vector value at index {i} is out of bounds.")
            self.face_blinding_factors.append(random_scalar())

        self.face_commitments, self.face_proofs = python_bulletproofs.prove_range_batch(
            shiftlist, self.face_blinding_factors, self.proof_bits)

        n = len(shiftlist)
        shifted_val_bytes = [int_to_scalar_bytes(v) for v in shiftlist]
        face_comms_raw = batch_pedersen_commit_raw(
            self.ptG, self.ptH, shifted_val_bytes, self.face_blinding_factors)
        k_x_list = [random_scalar() for _ in range(n)]
        k_r_list = [random_scalar() for _ in range(n)]

        # One merged 3n fixed-base call: rows 0..n-1 → R_i, rows n..2n-1 → A_R_i,
        # rows 2n..3n-1 → A_C_i = k_x*G + k_r*H.  One OpenMP launch instead of two.
        _ZERO = b'\x00' * 32
        fb = batch_multiscalar_mult_raw(
            [self.ptG, self.ptH],
            [[r,  _ZERO] for r  in self.face_blinding_factors] +
            [[kr, _ZERO] for kr in k_r_list] +
            [[kx, kr]    for kx, kr in zip(k_x_list, k_r_list)]
        )
        R_raws   = fb[:n]
        A_R_raws = fb[n:2*n]
        A_C_raws = fb[2*n:]

        # Parallel prove: encode×3 + SHA-512 + scalar_mul×2 per proof, all in OpenMP
        C_bytes_list = [bytes(c) for c in self.face_commitments]
        self.nizk_proofs = batch_nizk_hash_prove(
            A_C_raws, A_R_raws, R_raws, C_bytes_list,
            bytes(DALEK_G), bytes(DALEK_H),
            k_x_list, k_r_list,
            shifted_val_bytes, self.face_blinding_factors,
        )

        self.face_r_commitments = R_raws  # RawPoints — no encoding
        self.face_commitments_raw = face_comms_raw  # RawPoints — skip server-side decode

        return self.face_commitments, self.face_proofs, self.face_r_commitments, self.nizk_proofs, face_comms_raw

    def CreateDotCommitment(self) -> List[bytes]:
        """
        Step 2: Homomorphically computes the dot product of the face commitments 
        and the hypervectors, AND tracks the homomorphic blinders.
        """
        if not self.face_commitments:
            raise ValueError("[Client] Error: Run CommitFace() before CreateDotCommitment()!")
        
        # Curve order L for safely wrapping the blinder math
        L = (1 << 252) + 27742317777372353535851937790883648493

        blinding_ints = np.array([int.from_bytes(b, 'little') for b in self.face_blinding_factors], dtype=object)
        self.blind_dot_products = np.dot(self.hypervectors, blinding_ints)

        self.dot_blinding_factors = [int_to_scalar_bytes(int(v)) for v in self.blind_dot_products]
        CdfG = [int_to_scalar_bytes(int(v)) for v in self.raw_dot_products]
        CdfH = list(self.dot_blinding_factors)

        self.dot_commitments = batch_pedersen_commit_raw(self.ptG, self.ptH, CdfG, CdfH)
           

        return self.dot_commitments
    
    def CreateBitCommitment(self, GLOBAL_H: bytes) -> List[bytes]:
        """
        Step 3: Creates Pedersen commitments for the extracted LSH bits (0 or 1).
        Formula: C_bit = b_i * G + r_i * H
        """

        blist = []
        
        for b_val in self.hashed_bits:
            # 1. Generate a fresh, mathematically secure random scalar (r_i)
            r_i = random_scalar()
            self.bit_blinding_factors.append(r_i)
            
            # 2. Compute the value point: b_i * G
            # (If b_val is 0, this results in the identity point. If 1, it results in G)
            b_scalar = int_to_scalar_bytes(b_val)
            blist.append(b_scalar)
            
        self.bit_commitments = batch_pedersen_commit_raw(self.ptG,self.ptH,blist,self.bit_blinding_factors)
            
        return self.bit_commitments

    def CreateLinkageProof(self) -> Tuple[List[bytes], List[bytes]]:
        """
        Step 4: Computes C_test = C_d + 2^32 * C_b, and generates a 32-bit 
        Bulletproof to prove the sign bit correctly matches the dot product.
        """
        if not hasattr(self, 'dot_commitments') or not hasattr(self, 'bit_commitments'):
            raise ValueError("[Client] Error: Must compute dot and bit commitments first!")
        
        # Pre-compute the 2^32 shift
        scalar_shift_int = 1 << 32
        scalar_shift_bytes = int_to_scalar_bytes(scalar_shift_int)
        
        # The Curve25519 order (L) for safely wrapping blinders
        L = (1 << 252) + 27742317777372353535851937790883648493

        n = len(self.dot_commitments)

        # Batch variable-base: shift_i = 2^32 * C_b_i (all M in one OpenMP call)
        shift_points = batch_scalarmult_pairs_raw([scalar_shift_bytes] * n, list(self.bit_commitments))

        vtestlist = []
        r_test_list = []

        for i in range(n):
            C_test = point_add_raw(self.dot_commitments[i], shift_points[i])
            self.linkage_commitments.append(C_test)

            v_test = int(self.raw_dot_products[i]) + (self.hashed_bits[i] << 32)
            vtestlist.append(v_test)

            r_d_int = int.from_bytes(self.dot_blinding_factors[i], 'little')
            r_b_int = int.from_bytes(self.bit_blinding_factors[i], 'little')
            r_test_int = (r_d_int + scalar_shift_int * r_b_int) % L
            r_test_list.append(r_test_int.to_bytes(32, byteorder='little'))

        _, prooflist = python_bulletproofs.prove_range_batch(vtestlist, r_test_list, 32)

        self.linkage_proofs = prooflist

        return self.linkage_commitments, self.linkage_proofs
    

    def EvaluateOPRF(self, R_list: List[bytes], S0_list: List[int], S1_list: List[int], G_out: bytes) -> bytes:
        """
        Client Step 5: Evaluates the OT tuples to resolve the OPRF output without 
        revealing which bits were extracted. (Strictly Additive)
        """
        L = (1 << 252) + 27742317777372353535851937790883648493
        M = len(self.hashed_bits)
        
        obtained_sum = 0

        # Batch variable-base: shared_i = r_i * R_i (all M in one OpenMP call)
        shared_points = batch_scalarmult_pairs_raw(self.bit_blinding_factors, R_list)

        for i in range(M):
            shared_hash = hash_point_to_scalar(shared_points[i].encode())

            if self.hashed_bits[i] == 0:
                extracted_val = (S0_list[i] - shared_hash) % L
            else:
                extracted_val = (S1_list[i] - shared_hash) % L

            obtained_sum = (obtained_sum + extracted_val) % L
            
        # --- Final OPRF Output ---
        # Output = G_out + (obtained_sum * G)
        sum_scalar_bytes = int_to_scalar_bytes(obtained_sum)
        sum_point = self.ptG.scalarmult_raw(sum_scalar_bytes)
        
        final_oprf_point = point_add_raw(G_out, sum_point)
        
        self.oprf_output = final_oprf_point.encode()
        return self.oprf_output

### 2.2 FaceAuthenticationServer

Verifies client proofs and runs the OPRF evaluation.

In [ ]:
class FaceAuthenticationServer:
    def __init__(self, hypervectors: np.ndarray):
        """
        Initializes the server with the public LSH hypervectors.
        Sets up the exact same mathematical bounds as the client.
        """

        self.hypervectors = hypervectors
        
        # Cryptographic configuration (Must match client perfectly)
        self.data_bits = 12
        self.proof_bits = 16
        self.M = (1 << (self.data_bits - 1)) - 1  # 2047 shift for 12-bit
        
# -------------------------------------------- Server State Storage ------------------------------------------------------
        self.client_face_commitments = []
        self.dot_commitments = []
        self.bit_commitments =[]

        self.ptG = PrecompPoint(DALEK_G,expected_calls=10000)
        self.ptH = PrecompPoint(DALEK_H,expected_calls=10000)

        self.hypervector_scalars = [
        [int_to_scalar_bytes(int(v)) for v in row]
        for row in self.hypervectors
        ]

# -------------------------------------------- Server State Storage (end) ------------------------------------------------------

        # Initialize the OPRF Server's key and shifts
        self.k_0 = int.from_bytes(random_scalar(), 'little')
        num_hyperplanes = len(self.hypervectors)
        self.k_keys = [int.from_bytes(random_scalar(), 'little') for _ in range(num_hyperplanes)]


    def _nizk_batch_verify(self, commitments: List[bytes], r_commitments: List[bytes],
                            nizk_proofs: bytes,
                            commitments_raw=None) -> bool:
        """
        Single-challenge NIZK verifier for n statements (C = g^x*h^r, R = g^r).

        Correct Fiat-Shamir check: for each i, reconstruct
            Â_C_i = s_x·G + s_r·H − e·C_i
            Â_R_i = s_r·G − e·R_i
        then verify  e == H("JointPedR_v1" ‖ G ‖ H ‖ C_0 ‖ R_0 ‖ Â_C_0 ‖ Â_R_0 ‖ ... ‖ C_{n-1} ‖ R_{n-1} ‖ Â_C_{n-1} ‖ Â_R_{n-1}).

        All scalar mults are done in two parallel batch calls (precomputed G/H tables
        for fixed-base, one 2n variable-base call for C^{−e} and R^{−e}).
        """
        n = len(commitments)
        L = (1 << 252) + 27742317777372353535851937790883648493

        # --- Step 1: parse proof (single e + n response pairs) ---
        e_bytes  = bytes(nizk_proofs[:32])
        e_int    = int.from_bytes(e_bytes, 'little')
        sx_bytes = [bytes(nizk_proofs[32 + i*64 : 32 + i*64 + 32]) for i in range(n)]
        sr_bytes = [bytes(nizk_proofs[32 + i*64 + 32 : 32 + (i+1)*64]) for i in range(n)]

        C_raws      = (commitments_raw if commitments_raw is not None
                       else [decode_to_raw(bytes(c)) for c in commitments])
        neg_e_bytes = int_to_scalar_bytes((-e_int) % L)

        # --- Step 2: batch scalar mults (parallel, 8 cores) ---
        # Fixed-base: merge 2 calls into one 2n call — one OpenMP launch instead of two.
        # Rows 0..n-1 : g^{sx_i} · h^{sr_i}  (for Â_C)
        # Rows n..2n-1: g^{sr_i} · h^0 = g^{sr_i}  (for Â_R)
        _ZERO = b'\x00' * 32
        fixed_base = batch_multiscalar_mult_raw(
            [self.ptG, self.ptH],
            [[sx_bytes[i], sr_bytes[i]] for i in range(n)] +
            [[sr_bytes[i], _ZERO]       for i in range(n)]
        )
        gsx_hsr = fixed_base[:n]
        gsr     = fixed_base[n:]
        # Variable-base: C^{−e} and R^{−e} merged into one 2n call
        CR_neg_e = batch_scalarmult_pairs_raw(
            [neg_e_bytes] * (2 * n),
            C_raws + list(r_commitments)
        )
        C_neg_e = CR_neg_e[:n]
        R_neg_e = CR_neg_e[n:]

        # --- Step 3: reconstruct Â_C and Â_R (cheap additions, no scalar mults) ---
        A_hat_C = [point_add_raw(gsx_hsr[i], C_neg_e[i]) for i in range(n)]
        A_hat_R = [point_add_raw(gsr[i],     R_neg_e[i]) for i in range(n)]

        # --- Step 4: Fiat-Shamir hash check (parallel, OpenMP) ---
        # Encode R_i once for hashing (RawPoints → bytes)
        R_bytes_list = [r.encode() if hasattr(r, 'encode') else bytes(r)
                        for r in r_commitments]
        C_bytes_list = [bytes(c) for c in commitments]
        return batch_nizk_hash_verify(
            A_hat_C, A_hat_R,
            C_bytes_list, R_bytes_list,
            bytes(DALEK_G), bytes(DALEK_H),
            e_bytes,
        )

    def VerifyFaceCommitments(self, commitments: List[bytes], proofs: List[bytes],
                               r_commitments: List[bytes], nizk_proofs: bytes,
                               commitments_raw=None) -> bool:
        """
        Server Step 1: Verifies 16-bit Bulletproofs on the face commitments, then
        batch-verifies the NIZK proofs that C_i and R_i share the same blinding factor.
        """
        if len(commitments) != len(proofs):
            print("[Server] ERROR: Number of commitments does not match number of proofs.")
            return False

        print(f"\n[Server] Verifying {len(commitments)} Bulletproofs (16-bit capacity)...")
        results = python_bulletproofs.verify_range_batch(commitments, proofs, self.proof_bits)
        if not all(results):
            for i, ok in enumerate(results):
                if not ok:
                    print(f"[Server] REJECTED: Bulletproof at index {i} failed!")
            return False

        print(f"[Server] Batch-verifying {len(commitments)} NIZK proofs...")
        if not self._nizk_batch_verify(commitments, r_commitments, nizk_proofs, commitments_raw):
            print("[Server] REJECTED: NIZK batch verification failed!")
            return False

        self.client_face_commitments = commitments
        self.client_face_r_commitments = r_commitments
        return True
    
    def CreateDotCommitment(self) -> List[bytes]:
        """
        Server Step 2: Homomorphically computes the dot product of the verified 
        face commitments and the hypervectors, stripping the offset perfectly.
        """
        if not self.client_face_commitments:
            raise ValueError("[Server] Error: Must run VerifyFaceCommitments successfully first!")

        self.dot_commitments = []
    
        dotproductpointlist = batch_short_msm_auto_raw(self.hypervector_scalars,self.client_face_commitments)

        self.dot_commitments = batch_offset_subtract_raw(                        dotproductpointlist, self.hypervectors, self.M, self.ptG)  

        return self.dot_commitments
    
    def ReceiveBitCommitments(self, bit_commitments: List[bytes]):
        """
        Server Step 3: Receives the LSH bit commitments (C_b) from the client.
        The server does NOT verify these yet, it just stores them for the linkage step.
        """
        if len(bit_commitments) != len(self.hypervectors):
            raise ValueError("[Server] Error: Number of bit commitments does not match hyperplanes.")
            
        self.bit_commitments = bit_commitments

    def VerifyLinkageProofs(self, linkage_proofs: List[bytes]) -> bool:
        """
        Server Step 4: Homomorphically computes the linkage commitment C_test = C_d + 2^32 * C_b.
        Then, uses the client's provided 32-bit proofs to verify the math holds true.
        """
            
        if len(linkage_proofs) != len(self.dot_commitments):
            print("[Server] 🚨 ERROR: Number of proofs does not match number of commitments.")
            return False
        
        scalar_shift_bytes = int_to_scalar_bytes(1 << 32)
        n = len(self.dot_commitments)

        # Batch variable-base: shift_i = 2^32 * C_b_i (all n in one OpenMP call)
        shift_points = batch_scalarmult_pairs_raw([scalar_shift_bytes] * n, list(self.bit_commitments))

        # Build C_test = C_d + shift and encode for bulletproofs verification
        Ctestlist = [point_add_raw(self.dot_commitments[i], shift_points[i]).encode() for i in range(n)]

        results = python_bulletproofs.verify_range_batch(Ctestlist, linkage_proofs, 32)

        for i, is_valid in enumerate(results):
            if not is_valid:
                print(f"[Server] REJECTED: Zero-Knowledge Proof at index {i} failed!")
                return False

        return True
    
    def GenerateOPRFData(self, DALEK_H: bytes) -> tuple:
        M = len(self.bit_commitments)
        R_list, S0_list, S1_list = [], [], []
        
        # FIX: We use an additive sum instead of a product!
        delta_sum = 0
        G_raw = self.ptG.scalarmult_raw(int_to_scalar_bytes(1))

        # Pre-generate all random scalars upfront
        rou_list   = [random_scalar() for _ in range(M)]
        delta_list = [int.from_bytes(random_scalar(), 'little') for _ in range(M)]

        # Batch fixed-base: R_i = rou_i * H (one OpenMP launch, M ops)
        R_list = batch_multiscalar_mult_raw([self.ptH], [[r] for r in rou_list])

        # Pre-compute C_bi - G for each bit commitment
        C_bi_list         = list(self.bit_commitments)
        C_bi_minus_G_list = [point_sub_raw(C_bi, G_raw) for C_bi in C_bi_list]

        # Batch variable-base: P0_i = rou_i * C_bi, P1_i = rou_i * (C_bi - G)
        # Merged into one 2M call → single OpenMP launch
        P_all  = batch_scalarmult_pairs_raw(rou_list + rou_list, C_bi_list + C_bi_minus_G_list)
        P0_list = P_all[:M]
        P1_list = P_all[M:]

        # Sequential hash + scalar arithmetic (fast, no EC ops)
        for i in range(M):
            delta_i   = delta_list[i]
            delta_sum = (delta_sum + delta_i) % L

            hash0 = hash_point_to_scalar(P0_list[i].encode())
            S0_list.append((hash0 + delta_i) % L)

            hash1 = hash_point_to_scalar(P1_list[i].encode())
            S1_list.append((hash1 + delta_i + self.k_keys[i]) % L)
            
        # --- Prepare G_out = (k_0 - SUM(delta_i)) * G ---
        offset_scalar_int = (self.k_0 - delta_sum) % L
        G_out = self.ptG.scalarmult_raw(int_to_scalar_bytes(offset_scalar_int))
        
        return R_list, S0_list, S1_list, G_out
    
    def rawPRF(self, hash_int: int):
        L = (1 << 252) + 27742317777372353535851937790883648493
        M = len(self.k_keys)
        
        # Initialize the accumulator with the base key k_0
        accumulator = self.k_0
        
        for i in range(M):
            # Bitwise Magic: Shift the integer right by 'i' positions, 
            # then mask it with '& 1' to isolate exactly that bit (0 or 1).
            shift_amount = (M - 1) - i
            bit_is_set = (hash_int >> shift_amount) & 1       
            
            if bit_is_set == 1:
                # ADDITION ROUTE: We add the key to the running sum
                accumulator = (accumulator + self.k_keys[i]) % L
                
                # NOTE: If you are using the MULTIPLICATIVE route instead, 
                # change the initialization above to accumulator = self.k_0 
                # and use this line instead:
                # accumulator = (accumulator * self.k_keys[i]) % L

        # Map the final accumulated scalar to the curve using basepoint G
        prf_scalar_bytes = int_to_scalar_bytes(accumulator)
        final_prf_point = self.ptG.scalarmult(prf_scalar_bytes)
        
        return final_prf_point

### 2.3 Authenticated Encryption (AES-256-GCM)

In [ ]:
class AuthenticatedEncryption:
    """
    Implements the Authenticated Encryption scheme E (Enc, Dec)
    required for safPAKE using AES-256-GCM.
    """

    @staticmethod
    def derive_aes_key(ek: bytes) -> bytes:
        """Derives a 256-bit AES key from the OPRF output's ek using HKDF."""
        # Use a fixed salt and info string for deterministic key derivation
        # The key ek must be K bytes (32 bytes) long.
        salt = b'safpake_key_salt'
        info = b'safpake_aes_key_derivation'
        
        # HKDF is used to expand the PRF key 'ek' into a suitable 256-bit AES key
        return HKDF(
            algorithm=hashes.SHA256(),
            length=16, # AES-256 requires a 32-byte key
            salt=salt,
            info=info,
            backend=default_backend()
        ).derive(ek)

    @staticmethod
    def Encrypt(ek, data: bytes) -> bytes:
        """
        Encrypts data using AES-ECB.
        
        Args:
            data (bytes): The plaintext data to encrypt.
            
        Returns:
            bytes: The encrypted ciphertext.
        """
        key=AuthenticatedEncryption.derive_aes_key(ek)

        # We create a new cipher object for every operation to reset state
        cipher = AES.new(key, AES.MODE_ECB)
        
        # Pad the data to be a multiple of the block size (16 bytes)
        # standard PKCS7 padding is used here.
        padded_data = pad(data, AES.block_size)
        
        return cipher.encrypt(padded_data)

    @staticmethod
    def Decrypt(ek, ciphertext: bytes) -> bytes:
        """
        Decrypts data using AES-ECB.
        
        Args:
            ciphertext (bytes): The encrypted data to decrypt.
            
        Returns:
            bytes: The original plaintext.
        """
        key=AuthenticatedEncryption.derive_aes_key(ek)

        cipher = AES.new(key, AES.MODE_ECB)
        
        # Decrypt and then remove the padding
        padded_plaintext = cipher.decrypt(ciphertext)
        plaintext = unpad(padded_plaintext, AES.block_size)
        
        return plaintext

### 2.4 OPRF Key Derivation

In [ ]:
def generate_OPRF_keys(oprf_seed: bytes, bag_index: int, num_hyperplanes: int):
    """
    Deterministically generates the Server's Zero-Knowledge OPRF keys
    for a specific bag using the master OPRF seed.
    """
    k_keys = []
    
    # Simple KDF to derive 32-byte scalars safely
    def derive_scalar(seed, b_idx, counter):
        base = b"OPRF" + seed + b_idx.to_bytes(4, 'big') + counter.to_bytes(4, 'big')
        return hashlib.sha256(base).digest()

    # 1. Derive k_0
    k_0_bytes = derive_scalar(oprf_seed, bag_index, 0)
    k_0 = int.from_bytes(k_0_bytes, byteorder='little')
    
    # 2. Derive k_1 through k_M
    for i in range(1, num_hyperplanes + 1):
        k_i_bytes = derive_scalar(oprf_seed, bag_index, i)
        k_keys.append(int.from_bytes(k_i_bytes, byteorder='little'))
        
    return k_0, k_keys

### 2.5 PAKE_Bag — Single Disjoint Bag

In [ ]:
class PAKE_Bag:
    """
    Manages a single disjoint bag for a fuzzy-PAKE biometric authentication scheme.
    
    Args: 
        num_hyperplanes (int): The number of LSH bits (M).
        tolerance (int): The maximum Hamming distance allowed in the error ball.
    """

    def __init__(self, num_hyperplanes: int, tolerance: int, oprf_seed: bytes, bag_index: int, vec_dim: int = 512):
        self.DB = dict()
        self.AES_key_bytes = 32
        self.LSH_hash = CosineLSH(num_hyperplanes, vec_dim)
        self.tolerance = tolerance
        self.AE = AuthenticatedEncryption()
        self.bag_index = bag_index

        quantized_hyperplanes = CosineLSH.quantize_array(self.LSH_hash.hyperplanes, 12)
        self.bag_authenticator = FaceAuthenticationServer(quantized_hyperplanes)

        # --- NEW: Override the Server's random keys with our deterministic ones ---
        k_0, k_keys = generate_OPRF_keys(oprf_seed, bag_index, num_hyperplanes)
        self.bag_authenticator.k_0 = k_0
        self.bag_authenticator.k_keys = k_keys

        self.face_comms = None

    def derive_variants(self, orig_face_vec: np.ndarray, base_hash: int) -> list:
        """
        Derives all possible bit-flip combinations for the least reliable hyperplanes.
        """
        # 1. Ask the LSH which hyperplanes are too close to the boundary (using 12-bit quantization)
        indices = self.LSH_hash.filter_by_hyperplane(orig_face_vec, self.tolerance, k_bits=12)
        shifts = [int((self.LSH_hash.num_bits - 1) - idx) for idx in indices]
        
        base_template = base_hash
        for shift in shifts:
            base_template &= ~(1 << shift)  
            
        variants = []
        num_combinations = 1 << len(shifts) 
        for i in range(num_combinations):
            current_variant = base_template
            for j, shift in enumerate(shifts):
                if (i >> j) & 1:
                    current_variant |= (1 << shift)
            variants.append(current_variant)
        return variants
    
    def register(self, face_vector: np.ndarray, global_master_key: bytes):
        import time
        metrics = {}
        t_start_total = time.time()
        
        t0 = time.time()
        raw_base_hash = self.LSH_hash.hash(face_vector, k_bits=12)
        full_mask = (1 << self.LSH_hash.num_bits) - 1
        base_hash = raw_base_hash ^ full_mask
        t1 = time.time()
        metrics['setup_time'] = t1 - t0

        all_variants = self.derive_variants(face_vector, base_hash)
        t2 = time.time()
        metrics['derive_time'] = t2 - t1
        metrics['variant_count'] = len(all_variants)

        auth = self.bag_authenticator
        M    = len(auth.k_keys)
        N    = len(all_variants)
        payload_i = global_master_key

        # --- Step 1: Gray-code incremental scalar accumulation ---
        # variants[0] = base_template (no uncertain bits set).
        # Each successive Gray-code step flips exactly 1 uncertain bit,
        # so each PRF scalar costs 1 big-int add/sub instead of ~M/2.
        # Total: M (base init) + N-1 (incremental) vs N * M/2 operations.
        t_prf_start = time.time()

        indices       = self.LSH_hash.filtered_indices          # uncertain hyperplane indices
        uncertain_keys = [auth.k_keys[int(j)] for j in indices] # k_key for each uncertain bit

        base_acc = auth.k_0
        for k in range(M):
            if (all_variants[0] >> (M - 1 - k)) & 1:
                base_acc = (base_acc + auth.k_keys[k]) % L

        prf_scalars    = [None] * N
        current_acc    = base_acc
        prf_scalars[0] = int_to_scalar_bytes(current_acc)

        for i in range(1, N):
            gray_i    =  i      ^ (i      >> 1)
            gray_prev = (i - 1) ^ ((i - 1) >> 1)
            changed_j = (gray_i ^ gray_prev).bit_length() - 1  # which uncertain bit flipped
            if (gray_i >> changed_j) & 1:
                current_acc = (current_acc + uncertain_keys[changed_j]) % L
            else:
                current_acc = (current_acc - uncertain_keys[changed_j]) % L
            prf_scalars[i] = int_to_scalar_bytes(current_acc)

        # --- Step 2: One OpenMP batch for all N fixed-base scalar mults ---
        # batch_multiscalar_mult encodes inside the OpenMP loop — no separate encode step
        prf_bytes_list = batch_multiscalar_mult([auth.ptG], [[s] for s in prf_scalars])
        total_prf_time = time.time() - t_prf_start

        # --- Step 3: Hash + encrypt in parallel (OpenMP) ---
        t_enc_start = time.time()
        tags, ciphers = batch_hash_enc(prf_bytes_list, payload_i)
        for tag_bytes, cipher in zip(tags, ciphers):
            self.DB[tag_bytes.hex()] = cipher
        total_enc_time = time.time() - t_enc_start
            
        metrics['total_prf_time'] = total_prf_time
        metrics['total_enc_time'] = total_enc_time
        metrics['total_loop_time'] = time.time() - t2
        metrics['total_reg_time'] = time.time() - t_start_total
        
        return True, metrics

######----------------------------------------This Begins the Verification Routine ---------------------------------------------------------------
    
    def Check_Input(self, Authentication_Client, verified_face_comms) -> bool:
        """
        Executes Steps 2-4 of the ZKP protocol, relying on the global server 
        to have already validated the face commitments.
        """
        # =====================================================================
        # STEP 1: BYPASSED (Globally Verified)
        # =====================================================================
        # We inject the globally verified commitments directly into the authenticator
        self.bag_authenticator.client_face_commitments = verified_face_comms

        # =====================================================================
        # STEP 2: Homomorphic Dot Products
        # =====================================================================
        Authentication_Client.CreateDotCommitment()
        self.bag_authenticator.CreateDotCommitment()

        # =====================================================================
        # STEP 3: Bit Commitments
        # =====================================================================
        bit_comms = Authentication_Client.CreateBitCommitment(python_bulletproofs.get_dalek_default_h())
        self.bag_authenticator.ReceiveBitCommitments(bit_comms)

        # =====================================================================
        # STEP 4: Linkage Proofs (The 32-bit Shift)
        # =====================================================================
        linkage_comms, linkage_proofs = Authentication_Client.CreateLinkageProof()
        is_linkage_valid = self.bag_authenticator.VerifyLinkageProofs(linkage_proofs)

        if not is_linkage_valid:
            print("[Server] ❌ REJECTED: Linkage proofs failed. Bits do not match dot products.")
            return False

        return True

### 2.6 PAKE_server — Multi-Bag Server

In [ ]:
class PAKE_server:
    def __init__(self, disjoint_bags: int, number_of_hyperplanes: int, tolerance: int, vec_dim: int = 512):
        # Generate the global secrets once for the whole system
        self.oprf_seed = os.urandom(32)
        self.global_master_key = os.urandom(32)
        self.global_master_cipher = None
        self.AE = AuthenticatedEncryption()
        
        # 1. Create a bunch of disjoint bags, passing the seed and their index
        self.bags = []
        for i in range(disjoint_bags):
            bag = PAKE_Bag(number_of_hyperplanes, tolerance, self.oprf_seed, bag_index=i, vec_dim=vec_dim)
            self.bags.append(bag)
    
    def register(self, face_vec):
        import time
        t_global_start = time.time()
        
        # 1. Create the Global Master Cipher
        face_bytes = face_vec.astype(np.float32).tobytes()
        global_payload = face_bytes + self.oprf_seed
        self.global_master_cipher = self.AE.Encrypt(self.global_master_key, global_payload)
        
        result = True
        total_metrics = {}
        
        # 2. Register on each bag and accumulate ALL metrics
        for each in self.bags:
            bag_success, metrics = each.register(face_vec, self.global_master_key)
            result = result and bag_success
            
            for key, value in metrics.items():
                total_metrics[key] = total_metrics.get(key, 0.0) + value
                
        # 3. Add the total server registration time (including master cipher creation)
        total_metrics['server_total_reg_time'] = time.time() - t_global_start
        
        return result, total_metrics
    
    def authenticate(self, client):
        # The Client now orchestrates the bag looping and multi-bag logic internally!
        return client.verify(self)
    
    def verify_global_face(self, face_comms, face_proofs, face_r_comms, nizk_proofs, face_comms_raw=None):
        """
        GLOBAL OPTIMIZATION: Verifies the 128 Face Commitments exactly once 
        for the entire system, preventing redundant Bulletproof checks.
        """
        print("\n[Global Server] Verifying User Face Commitments...")
        return self.bags[0].bag_authenticator.VerifyFaceCommitments(face_comms, face_proofs, face_r_comms, nizk_proofs, face_comms_raw)

### 2.7 PAKE_client

In [ ]:
class PAKE_client:
    """Creates a single client for a single PAKE disjoint bag."""
    
    def __init__(self, face_vec: np.ndarray):
        # 1. Quantize the Client's Face Vector to 12 bits

        self.face = CosineLSH.quantize_array(face_vec, 12)

        # These are not yet initialized.
        # The Server will send its public LSH parameters to the Client to initialize.
        self.LSH_hash = None
        self.Face_Authenticator = None # this is the previous authentication client class
        self.sessonOPRF=None
    
    def Initialize(self, LSH):
        self.LSH_hash = LSH
        quantized_hyperplanes = CosineLSH.quantize_array(LSH.hyperplanes, 12)
        
        if self.Face_Authenticator is None:
            # First initialization: Set up the whole ZKP backend
            self.Face_Authenticator = FaceAuthenticationClient(self.face, quantized_hyperplanes)
        else:
            # REUSE OPTIMIZATION: Keep the face commitments, just update the bag data!
            self.Face_Authenticator.hypervectors = quantized_hyperplanes
            
            # Recalculate plaintext dots and bits for THIS specific bag
            self.Face_Authenticator.raw_dot_products = np.dot(quantized_hyperplanes, self.Face_Authenticator.facial_vector)
            self.Face_Authenticator.hashed_bits = [0 if dot >= 0 else 1 for dot in self.Face_Authenticator.raw_dot_products]
            
            # Clear out the old bag's intermediate commitments
            self.Face_Authenticator.dot_commitments = []
            self.Face_Authenticator.dot_blinding_factors = []
            self.Face_Authenticator.bit_commitments = []
            self.Face_Authenticator.bit_blinding_factors = []
            self.Face_Authenticator.linkage_commitments = []
            self.Face_Authenticator.linkage_proofs = []

    def getOPRF(self, Authentication_Bag, verified_face_comms):
        if not hasattr(self, 'metrics'): self.metrics = {}
        prefix = "audit_" if getattr(self, 'is_auditing', False) else "init_"

        # 1. ZKP Verification Time (Now only tracks Steps 2-4)
        t0 = time.time()
        is_verified = Authentication_Bag.Check_Input(self.Face_Authenticator, verified_face_comms)
        t1 = time.time()
        self.metrics[prefix + 'zkp_time'] = self.metrics.get(prefix + 'zkp_time', 0.0) + (t1 - t0)

        if not is_verified: return False

        Authentication_server = Authentication_Bag.bag_authenticator
        
        # 2. Server OT Generation
        t2 = time.time()
        R_list, S0_list, S1_list, G_out = Authentication_server.GenerateOPRFData(python_bulletproofs.get_dalek_default_h())
        t3 = time.time()
        self.metrics[prefix + 'server_ot_gen_time'] = self.metrics.get(prefix + 'server_ot_gen_time', 0.0) + (t3 - t2)

        # 3. Client OT Evaluation
        t4 = time.time()
        oprf_point_bytes = self.Face_Authenticator.EvaluateOPRF(R_list, S0_list, S1_list, G_out)
        t5 = time.time()
        self.metrics[prefix + 'client_ot_eval_time'] = self.metrics.get(prefix + 'client_ot_eval_time', 0.0) + (t5 - t4)

        self.sessonOPRF = oprf_point_bytes

    def derive_variants(self, registered_face) -> list:
        if self.LSH_hash is None:
            raise RuntimeError("[Client] Cannot derive variants: LSH is not initialized.")

        # 1. Calculate the Client's current base_hash
        raw_base_hash = self.LSH_hash.hash(registered_face, k_bits=12)
        
        # FIX: Invert the hash to match the ZKP "Sign-Bit" logic
        full_mask = (1 << self.LSH_hash.num_bits) - 1
        base_hash = raw_base_hash ^ full_mask

        # 2. EXPLICIT FIX: Run the filter to populate the indices!
        # self.LSH_hash.filter_by_hyperplane(registered_face, self.LSH_hash.tolerance, k_bits=12)
        # I commented out the previous one since the indices should already be initialized
        indices = self.LSH_hash.filtered_indices 
        
        # 3. Reverse the index because CosineLSH hashes MSB-first
        shifts = [int((self.LSH_hash.num_bits - 1) - idx) for idx in indices]
        
        # 4. Prepare the base template: Clear the bits at the unreliable positions
        base_template = base_hash
        for shift in shifts:
            base_template &= ~(1 << shift)  
            
        variants = []
        num_combinations = 1 << len(shifts) 

        # 5. Generate all combinations using a binary counter
        for i in range(num_combinations):
            current_variant = base_template
            for j, shift in enumerate(shifts):
                # If the j-th bit of our counter 'i' is 1, set the bit in the variant
                if (i >> j) & 1:
                    current_variant |= (1 << shift)
                    
            variants.append(current_variant)

        return variants
    
    def get_Oprf_Keys(self, recoveredMasterKey: bytes, master_cipher: bytes):
        """
        Decrypts the Master Ciphertext and parses out the Server's persistent 
        Zero-Knowledge OPRF keys (k_0 and k_1 ... k_M).
        """
        # (Assuming AuthenticatedEncryption is available in the Client's scope)
        AE = AuthenticatedEncryption()
        
        try:
            # 1. Decrypt the heavy envelope payload
            server_keys_payload = AE.Decrypt(recoveredMasterKey, master_cipher)
        except Exception as e:
            raise RuntimeError("[Client] ❌ Failed to decrypt Master Ciphertext! Invalid Master Key.")

        # 2. Extract k_0 (The first 32 bytes)
        k_0 = int.from_bytes(server_keys_payload[0:32], byteorder='little')
        
        # 3. Extract the remaining M keys (32 bytes each)
        M = self.LSH_hash.num_bits
        k_keys = []
        
        offset = 32
        for i in range(M):
            chunk = server_keys_payload[offset : offset + 32]
            k_i = int.from_bytes(chunk, byteorder='little')
            k_keys.append(k_i)
            offset += 32
        
        return k_0, k_keys
    
    def RawOPRF(self,k_0,k_keys, pw_variant:int):
        
        """
        Locally computes the PRF output for a given integer variant.
        Mirrors the Server's direct evaluation to independently audit the DB.
        """
        # The Curve25519 Prime Order
        L = (1 << 252) + 27742317777372353535851937790883648493
        M = self.LSH_hash.num_bits
        
        # Start the accumulator at the base key k_0
        accumulator = k_0
        
        for i in range(M):
            # Read the bits exactly how the server read them!
            # (If your server used the standard LSB-first mapping):
            shift_amount = (M - 1) - i
            bit_is_set = (pw_variant >> shift_amount) & 1   
            
            if bit_is_set == 1:
                # ADDITION ROUTE: Add the corresponding key modulo L
                accumulator = (accumulator + k_keys[i]) % L

        # Map the final accumulated scalar back to a geometric point on the curve
        # (Assuming int_to_scalar_bytes and scalar_to_point are in your global scope)
        prf_scalar_bytes = int_to_scalar_bytes(accumulator)
        final_prf_point = scalar_to_point(prf_scalar_bytes)
        
        return final_prf_point
    
    def verify(self, main_server):  
        import time
        self.metrics = {}
        self.is_auditing = False
        t_start_auth = time.time()

        # =========================================================
        # GLOBAL OPTIMIZATION: Commit to Face ONCE
        # =========================================================
        # Initialize the backend using the first bag to load the arrays
        self.Initialize(main_server.bags[0].LSH_hash)
        
        t_face_zkp_start = time.time()
        face_comms, face_proofs, face_r_comms, nizk_proofs, face_comms_raw = self.Face_Authenticator.CommitFace()
        is_face_valid = main_server.verify_global_face(face_comms, face_proofs, face_r_comms, nizk_proofs, face_comms_raw)
        self.metrics['init_global_face_zkp_time'] = time.time() - t_face_zkp_start
        
        if not is_face_valid:
            print("[Client] ❌ Authentication aborted. Global Face ZKP failed.")
            return False

        oprf_results = []
        
        # 1. Interactive Phase: Execute ZKP & OT with EVERY bag
        for server_bag in main_server.bags:
            self.Initialize(server_bag.LSH_hash)
            # Pass the globally verified commitments directly into the bag check
            self.getOPRF(server_bag, face_comms)
            oprf_results.append(self.sessonOPRF)

        # ... (The rest of verify() remains exactly the same starting from DB Lookup) ...
        recovered_master_key = None
        AE = AuthenticatedEncryption()
        
        for bag, oprf_point in zip(main_server.bags, oprf_results):
            if not oprf_point: continue
                
            tag_str = hashlib.sha256(b"TAG" + oprf_point).digest().hex()
            key_i   = hashlib.sha256(b"KEY" + oprf_point).digest()

            if tag_str in bag.DB:
                try:
                    recovered_master_key = secretbox_open(key_i, bag.DB[tag_str])
                    break
                except ValueError:
                    continue

        if not recovered_master_key:
            print("[Client] ❌ Authentication Failed: Face vector is outside all error balls.")
            return False

        try:
            global_payload = AE.Decrypt(recovered_master_key, main_server.global_master_cipher)
        except Exception:
            print("[Client] ❌ Decryption Failed: Invalid Global Master Cipher.")
            return False

        self.oprf_seed = global_payload[-32:]
        recovered_face_bytes = global_payload[:-32]
        self.global_master_key = recovered_master_key
        self.original_face_vector = np.frombuffer(recovered_face_bytes, dtype=np.float32)
        self.metrics['init_total_auth_time'] = time.time() - t_start_auth

        t_start_audit = time.time()
        audit_success = self.audit_all_bags(main_server)
        self.metrics['total_global_audit_time'] = time.time() - t_start_audit
        
        return audit_success

    def audit_all_bags(self, pake_server):
        """
        Phase 2: Local non-interactive audit. Deterministically generates the OPRF 
        keys for every bag to audit the error balls without any Zero-Knowledge Proofs.
        """
        import time
        if getattr(self, 'original_face_vector', None) is None or not hasattr(self, 'oprf_seed'):
            print("[Client] ❌ Cannot audit server: Missing face vector or OPRF seed.")
            return False
        
        self.is_auditing = True
        self.metrics['audit_prf_time'] = 0.0
        self.metrics['audit_kdf_decrypt_time'] = 0.0
        self.metrics['audit_derive_variants_time'] = 0.0
        
        AE = AuthenticatedEncryption()
        
        for idx, server_bag in enumerate(pake_server.bags):
            # Tell the client which LSH parameters to use for this bag's math
            self.LSH_hash = server_bag.LSH_hash
            
            # --- HUGE UPGRADE: Deterministic Keys ---
            # We skip the ZKP/OT entirely and just generate the keys locally!
            # (Assuming the generate_OPRF_keys function from earlier is in scope)
            OPRF_k0, OPRF_keys = generate_OPRF_keys(self.oprf_seed, idx, server_bag.LSH_hash.num_bits)
            
            # Profile Variant Derivation
            t_derive_start = time.time()
            variants = self.derive_variants(self.original_face_vector)
            self.metrics['audit_derive_variants_time'] += (time.time() - t_derive_start)
            
            M = server_bag.LSH_hash.num_bits
            N = len(variants)

            # --- Batch PRF: Gray-code incremental accumulation + OpenMP EC mult ---
            t_prf_start = time.time()

            indices        = self.LSH_hash.filtered_indices
            uncertain_keys = [OPRF_keys[int(j)] for j in indices]

            base_acc = OPRF_k0
            for k in range(M):
                if (variants[0] >> (M - 1 - k)) & 1:
                    base_acc = (base_acc + OPRF_keys[k]) % L

            prf_scalars    = [None] * N
            current_acc    = base_acc
            prf_scalars[0] = int_to_scalar_bytes(current_acc)

            for i in range(1, N):
                gray_i    =  i      ^ (i      >> 1)
                gray_prev = (i - 1) ^ ((i - 1) >> 1)
                changed_j = (gray_i ^ gray_prev).bit_length() - 1
                if (gray_i >> changed_j) & 1:
                    current_acc = (current_acc + uncertain_keys[changed_j]) % L
                else:
                    current_acc = (current_acc - uncertain_keys[changed_j]) % L
                prf_scalars[i] = int_to_scalar_bytes(current_acc)

            # batch_multiscalar_mult encodes inside the OpenMP loop — no separate encode step
            ptG_bag        = server_bag.bag_authenticator.ptG
            prf_bytes_list = batch_multiscalar_mult([ptG_bag], [[s] for s in prf_scalars])
            self.metrics['audit_prf_time'] += time.time() - t_prf_start

            # --- KDF + decrypt in parallel (OpenMP) ---
            t_kdf_start = time.time()
            expected_payload = self.global_master_key
            # Step 1: compute all tags and keys in parallel
            tags, keys = batch_sha256_tag_key(prf_bytes_list)
            # Step 2: gather ciphertexts from DB (fast Python hash-map lookups)
            ciphers = []
            for tag_bytes in tags:
                tag_str = tag_bytes.hex()
                if tag_str not in server_bag.DB:
                    raise RuntimeError(f"[Client] 🚨 AUDIT FAILED: Server omitted variant in Bag {idx}!")
                ciphers.append(server_bag.DB[tag_str])
            # Step 3: decrypt and verify all in parallel
            if not batch_secretbox_open_verify(keys, ciphers, expected_payload):
                raise RuntimeError(f"[Client] 🚨 AUDIT FAILED: Tampered payload in Bag {idx}!")
            self.metrics['audit_kdf_decrypt_time'] += time.time() - t_kdf_start

        self.is_auditing = False
        return True

## 3. Helper Utilities

In [ ]:
def quantize_to_12bit(float_vector: np.ndarray) -> np.ndarray:
    """
    Safely scales a float vector [-1.0, 1.0] to a 12-bit signed integer vector [-2047, 2047].
    """
    # Clip to ensure no outliers break the math
    clipped = np.clip(float_vector, -1.0, 1.0)
    # Scale by 2047 and round to nearest integer
    quantized = np.round(clipped * 2047).astype(int)
    return quantized

VECTOR_DIM = 512
NUM_HYPERPLANES = 64

## 4. Data Loading

Load face embeddings from the VGGFace2 dataset extracted with the buffalo_l (ResNet50/ArcFace)
model via InsightFace. Each person maps to a list of 512-D unit vectors.

The JSON file is not included in this repository (it contains extracted biometric data).
To reproduce: run the extraction cell in the original notebook, or adapt the code below to
your own embedding dataset. The expected format is:
```json
{ "person_id": [[...512 floats...], [...], ...], ... }
```


In [ ]:
DATA_FILENAME = "data_train_resnet50.json"

# Search for the data file in the current directory and common sibling locations
_search_paths = [
    DATA_FILENAME,
    os.path.join('..', 'Biometric-PAKE', DATA_FILENAME),
]
DATA_FILE = next((p for p in _search_paths if os.path.exists(p)), None)

if DATA_FILE:
    with open(DATA_FILE, "r") as f:
        facial_data_loaded = json.load(f)
    facial_data = {k: [np.array(emb) for emb in v] for k, v in facial_data_loaded.items()}
    print(f"Loaded {len(facial_data)} identities from {DATA_FILE}.")
else:
    facial_data = {}
    print(f"[WARNING] {DATA_FILENAME} not found. Benchmarks requiring facial data will not run.")


## 5. Benchmarks

### 5.1 EC Microbenchmarks

In [ ]:
def run_microbenchmarks(iterations: int = 30) -> pd.DataFrame:
    """
    Measures the average execution time of libsodium EC wrapper functions.
    """
    print(f"=== Starting Microbenchmarks ({iterations} iterations per function) ===")
    
    # 1. Pre-generate valid inputs so we don't accidentally time the setup
    # (Assuming your wrappers are already loaded and working)
    s1 = random_scalar()
    s2 = random_scalar()
    p1 = random_point()
    p2 = random_point()
    
    # Ristretto255 operates on 32-byte chunks, so we create 32 random bytes for XOR
    b1 = os.urandom(32)
    b2 = os.urandom(32)

    # 2. Define the functions to test and their required arguments
    tests = [
        ("random_scalar", random_scalar, ()),
        ("Fixed-Based Multiplication", scalar_to_point, (s1,)),
        ("Variable-Based Multiplication", point_mul, (s1, p1)),
        ("EC Addition", point_add, (p1, p2)),
        ("EC Subtraction", point_sub, (p1, p2))
    ]

    results = []

    # 3. Run the benchmarks
    for func_name, func, args in tests:
        times = []
        for _ in range(iterations):
            start = time.perf_counter()
            func(*args)
            end = time.perf_counter()
            
            times.append(end - start)
            
        # Calculate the average time
        avg_time_sec = sum(times) / iterations
        
        results.append({
            "Function": func_name,
            "Avg Time (Seconds)": avg_time_sec,
            "Avg Time (Microseconds)": avg_time_sec * 1_000_000
        })

    # 4. Format into a pandas DataFrame and sort by slowest to fastest
    df = pd.DataFrame(results)
    df = df.sort_values(by="Avg Time (Microseconds)", ascending=False).reset_index(drop=True)
    
    return df

### 5.2 ZK-LSH Benchmark (without OPRF)

In [ ]:
def run_zklsh_benchmark(facial_data_dict: dict, num_iterations: int = 10, output_file: str = "BulletProof_Routine_benchmarks.csv"):
    print(f"\n🚀 Starting Zero-Knowledge LSH Benchmark ({num_iterations} iterations)...")
    
    VECTOR_DIM = 512
    NUM_HYPERPLANES = 64
    
    # We will track timings in a dictionary of lists
    timings = {
        "Client_1_CommitFace": [],
        "Server_1_VerifyFace": [],
        "Client_2_DotCommitment": [],
        "Server_2_DotCommitment": [],
        "Client_3_BitCommitment": [],
        "Server_3_ReceiveBits": [],
        "Client_4_LinkageProof": [],
        "Server_4_VerifyLinkage": [],
        "Total_Client_Time": [],
        "Total_Server_Time": [],
        "Total_Protocol_Time": []
    }

    person_keys = list(facial_data_dict.keys())

    for i in range(num_iterations):
        print(f"   -> Running iteration {i+1}/{num_iterations}...")
        
        # --- 1. Fresh Data Setup ---
        # Grab a random person and a random face from their array
        random_person = random.choice(person_keys)
        raw_face = random.choice(facial_data_dict[random_person])
        raw_hyper = np.random.uniform(-1.0, 1.0, (NUM_HYPERPLANES, VECTOR_DIM))
        
        quantized_face = quantize_to_12bit(raw_face)
        quantized_hyper = quantize_to_12bit(raw_hyper)
        
        client = FaceAuthenticationClient(quantized_face, quantized_hyper)
        server = FaceAuthenticationServer(quantized_hyper)
        
        client_time_total = 0.0
        server_time_total = 0.0

        # --- STEP 1 ---
        t0 = time.perf_counter()
        face_comms, face_proofs, face_r_comms, nizk_proofs, face_comms_raw = client.CommitFace()
        t_client1 = time.perf_counter() - t0
        client_time_total += t_client1
        timings["Client_1_CommitFace"].append(t_client1)

        t0 = time.perf_counter()
        is_face_valid = server.VerifyFaceCommitments(face_comms, face_proofs, face_r_comms, nizk_proofs, face_comms_raw)
        t_server1 = time.perf_counter() - t0
        server_time_total += t_server1
        timings["Server_1_VerifyFace"].append(t_server1)
        
        if not is_face_valid: raise RuntimeError("Protocol Failed at Step 1.")

        # --- STEP 2 ---
        t0 = time.perf_counter()
        client.CreateDotCommitment()
        t_client2 = time.perf_counter() - t0
        client_time_total += t_client2
        timings["Client_2_DotCommitment"].append(t_client2)

        t0 = time.perf_counter()
        server.CreateDotCommitment()
        t_server2 = time.perf_counter() - t0
        server_time_total += t_server2
        timings["Server_2_DotCommitment"].append(t_server2)

        # --- STEP 3 ---
        t0 = time.perf_counter()
        bit_comms = client.CreateBitCommitment(DALEK_H)
        t_client3 = time.perf_counter() - t0
        client_time_total += t_client3
        timings["Client_3_BitCommitment"].append(t_client3)

        t0 = time.perf_counter()
        server.ReceiveBitCommitments(bit_comms)
        t_server3 = time.perf_counter() - t0
        server_time_total += t_server3
        timings["Server_3_ReceiveBits"].append(t_server3)

        # --- STEP 4 ---
        t0 = time.perf_counter()
        linkage_comms, linkage_proofs = client.CreateLinkageProof()
        t_client4 = time.perf_counter() - t0
        client_time_total += t_client4
        timings["Client_4_LinkageProof"].append(t_client4)

        t0 = time.perf_counter()
        is_linkage_valid = server.VerifyLinkageProofs(linkage_proofs)
        t_server4 = time.perf_counter() - t0
        server_time_total += t_server4
        timings["Server_4_VerifyLinkage"].append(t_server4)
        
        if not is_linkage_valid: raise RuntimeError("Protocol Failed at Step 4.")

        # --- Totals ---
        timings["Total_Client_Time"].append(client_time_total)
        timings["Total_Server_Time"].append(server_time_total)
        timings["Total_Protocol_Time"].append(client_time_total + server_time_total)

    # --- 4. Process and Export to CSV ---
    print(f"\n📊 Benchmarking complete! Exporting results to {output_file}...")
    
    with open(output_file, mode='w', newline='') as file:
        writer = csv.writer(file)
        # Write CSV Headers
        writer.writerow(["Operation", "Mean_Time_(s)", "Std_Dev_(s)", "Min_Time_(s)", "Max_Time_(s)"])
        
        for operation, times in timings.items():
            mean_time = statistics.mean(times)
            std_dev = statistics.stdev(times) if len(times) > 1 else 0.0
            min_time = min(times)
            max_time = max(times)
            
            writer.writerow([
                operation, 
                f"{mean_time:.6f}", 
                f"{std_dev:.6f}", 
                f"{min_time:.6f}", 
                f"{max_time:.6f}"
            ])
            
            # Print to console for immediate satisfaction
            print(f"{operation.ljust(30)} | Mean: {mean_time:.4f}s")

    print(f"\n✅ All done. Open '{output_file}' to view the detailed metrics.")

### 5.3 Full Protocol Benchmark (with OPRF)

In [ ]:
def run_full_protocol_benchmark(facial_data_dict: dict, num_iterations: int = 10, output_file: str = "full_protocol_benchmarks.csv"):

    VECTOR_DIM = 512
    NUM_HYPERPLANES = 64
    
    # Track timings for all 5 stages
    timings = {
        "Client_1_CommitFace": [],
        "Server_1_VerifyFace": [],
        "Client_2_DotCommitment": [],
        "Server_2_DotCommitment": [],
        "Client_3_BitCommitment": [],
        "Server_3_ReceiveBits": [],
        "Client_4_LinkageProof": [],
        "Server_4_VerifyLinkage": [],
        "Server_5_GenerateOPRF": [],  # NEW: OT Setup
        "Client_5_EvaluateOPRF": [],  # NEW: OT Eval
        "Total_Client_Time": [],
        "Total_Server_Time": [],
        "Total_Protocol_Time": []
    }

    person_keys = list(facial_data_dict.keys())

    for i in range(num_iterations):
        print(f"\n   ---> Running iteration {i+1}/{num_iterations}...")
        
        # --- 1. Fresh Data Setup ---
        random_person = random.choice(person_keys)
        person_faces = facial_data_dict[random_person]
        random_face_idx = random.randint(0, len(person_faces) - 1)
        
        raw_face = np.array(person_faces[random_face_idx]).flatten()

        raw_hyper = np.random.uniform(-1.0, 1.0, (NUM_HYPERPLANES, VECTOR_DIM))
        quantized_face = quantize_to_12bit(raw_face)
        quantized_hyper = quantize_to_12bit(raw_hyper)
        
        client = FaceAuthenticationClient(quantized_face, quantized_hyper)
        server = FaceAuthenticationServer(quantized_hyper)
        
        client_time_total = 0.0
        server_time_total = 0.0

        # --- STEP 1: Face Commitments ---
        t0 = time.perf_counter()
        face_comms, face_proofs, face_r_comms, nizk_proofs, face_comms_raw = client.CommitFace()
        client_time_total += (time.perf_counter() - t0); timings["Client_1_CommitFace"].append(time.perf_counter() - t0)

        t0 = time.perf_counter()
        is_face_valid = server.VerifyFaceCommitments(face_comms, face_proofs, face_r_comms, nizk_proofs, face_comms_raw)
        server_time_total += (time.perf_counter() - t0); timings["Server_1_VerifyFace"].append(time.perf_counter() - t0)
        if not is_face_valid: raise RuntimeError("Failed at Step 1.")

        # --- STEP 2: Dot Products ---
        t0 = time.perf_counter()
        client.CreateDotCommitment()
        client_time_total += (time.perf_counter() - t0); timings["Client_2_DotCommitment"].append(time.perf_counter() - t0)

        t0 = time.perf_counter()
        server.CreateDotCommitment()
        server_time_total += (time.perf_counter() - t0); timings["Server_2_DotCommitment"].append(time.perf_counter() - t0)

        # --- STEP 3: Bit Commitments ---
        t0 = time.perf_counter()
        bit_comms = client.CreateBitCommitment(DALEK_H)
        client_time_total += (time.perf_counter() - t0); timings["Client_3_BitCommitment"].append(time.perf_counter() - t0)

        t0 = time.perf_counter()
        server.ReceiveBitCommitments(bit_comms)
        server_time_total += (time.perf_counter() - t0); timings["Server_3_ReceiveBits"].append(time.perf_counter() - t0)

        # --- STEP 4: Linkage Proofs ---
        t0 = time.perf_counter()
        linkage_comms, linkage_proofs = client.CreateLinkageProof()
        client_time_total += (time.perf_counter() - t0); timings["Client_4_LinkageProof"].append(time.perf_counter() - t0)

        t0 = time.perf_counter()
        is_linkage_valid = server.VerifyLinkageProofs(linkage_proofs)
        server_time_total += (time.perf_counter() - t0); timings["Server_4_VerifyLinkage"].append(time.perf_counter() - t0)
        if not is_linkage_valid: raise RuntimeError("Failed at Step 4.")

        # --- STEP 5: OT & OPRF Phase (NEW) ---
        t0 = time.perf_counter()
        R_list, S0_list, S1_list, G_out = server.GenerateOPRFData(DALEK_H)
        server_time_total += (time.perf_counter() - t0); timings["Server_5_GenerateOPRF"].append(time.perf_counter() - t0)

        t0 = time.perf_counter()
        final_oprf_point = client.EvaluateOPRF(R_list, S0_list, S1_list, G_out)
        client_time_total += (time.perf_counter() - t0); timings["Client_5_EvaluateOPRF"].append(time.perf_counter() - t0)
        
        # Print the final resulting point (just the first 16 hex chars to keep the console clean)
        print(f"      💎 Final OPRF Output: {final_oprf_point.hex()[:16]}...{final_oprf_point.hex()[-16:]}")

        # --- Totals ---
        timings["Total_Client_Time"].append(client_time_total)
        timings["Total_Server_Time"].append(server_time_total)
        timings["Total_Protocol_Time"].append(client_time_total + server_time_total)

    # --- Process and Export to CSV ---
    print(f"\n📊 Benchmarking complete! Exporting results to {output_file}...")
    
    with open(output_file, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(["Operation", "Mean_Time_(s)", "Mean_Time_(ms)", "Std_Dev_(s)", "Min_Time_(s)", "Max_Time_(s)"])
        
        for operation, times in timings.items():
            mean_time = statistics.mean(times)
            mean_ms = mean_time * 1000  # Convert to milliseconds for easier reading
            std_dev = statistics.stdev(times) if len(times) > 1 else 0.0
            min_time = min(times)
            max_time = max(times)
            
            writer.writerow([
                operation, 
                f"{mean_time:.6f}", 
                f"{mean_ms:.3f}", 
                f"{std_dev:.6f}", 
                f"{min_time:.6f}", 
                f"{max_time:.6f}"
            ])
            
            print(f"{operation.ljust(30)} | Mean: {mean_time:.4f}s ({mean_ms:.1f} ms)")

    print(f"\n✅ All done. Detailed metrics saved to '{output_file}'.")

### 5.4 PAKE Profiling Sweep

In [ ]:
def run_pake_profiling(num_bits: int, tolerance: int, disjoint_bags: int = 3, num_trials: int = 5, filename: str = "pake_profiling.csv"):
    """
    Executes a full end-to-end PAKE protocol multiple times, averages the micro-metrics, 
    and appends the smoothed results to a CSV file.
    """
    print(f"\n==================================================")
    print(f"🚀 Profiling PAKE (Bits: {num_bits}, Tol: {tolerance}, Bags: {disjoint_bags}, Trials: {num_trials})")
    print(f"==================================================")

    variants_per_bag = 1 << tolerance
    total_variants = disjoint_bags * variants_per_bag

    # Accumulators for our metrics
    accumulated_reg = {}
    accumulated_auth = {}
    accumulated_session_time = 0.0
    successful_trials = 0

    person_keys = list(facial_data.keys())

    for trial in range(1, num_trials + 1):
        print(f"[*] Executing Trial {trial}/{num_trials}...")
        
        # 1. Setup Fresh Dummy Data for this trial
        random_person = random.choice(person_keys)
        raw_face = random.choice(facial_data[random_person])

        # ---------------------------------------------------------
        # 2. SETUP SERVER & PROFILE REGISTRATION
        # ---------------------------------------------------------
        server = PAKE_server(disjoint_bags, number_of_hyperplanes=num_bits, tolerance=tolerance)
        
        # We get the success boolean and the ACCUMULATED micro-metrics from all bags
        reg_success, reg_metrics = server.register(raw_face)
        
        if not reg_success:
            print(f"    ⚠️ Trial {trial} Server Registration failed. Skipping.")
            continue

        # ---------------------------------------------------------
        # 3. PROFILE AUTHENTICATION & AUDIT
        # ---------------------------------------------------------
        client = PAKE_client(raw_face)
        
        t_total_start = time.time()
        auth_success = server.authenticate(client)
        t_total_end = time.time()

        if not auth_success:
            print(f"    ⚠️ Trial {trial} Authentication failed. Skipping.")
            continue

        # ---------------------------------------------------------
        # 4. ACCUMULATE METRICS
        # ---------------------------------------------------------
        auth_metrics = client.metrics
        
        for key, value in reg_metrics.items():
            accumulated_reg[key] = accumulated_reg.get(key, 0.0) + value
            
        for key, value in auth_metrics.items():
            accumulated_auth[key] = accumulated_auth.get(key, 0.0) + value
            
        accumulated_session_time += (t_total_end - t_total_start)
        successful_trials += 1

    # =========================================================
    # 5. CALCULATE AVERAGES & WRITE TO CSV
    # =========================================================
    if successful_trials == 0:
        print("❌ All trials failed. No data recorded.")
        return False

    # Compute FP/FN rates
    print(f"[*] Computing FP/FN rates...")
    fp_rate, fn_rate = compute_fp_fn_rates(
        num_bits=num_bits, tolerance=tolerance, disjoint_bags=disjoint_bags,
        facial_data=facial_data
    )

    # Calculate the averages
    avg_reg = {k: v / successful_trials for k, v in accumulated_reg.items()}
    avg_auth = {k: v / successful_trials for k, v in accumulated_auth.items()}
    avg_session = accumulated_session_time / successful_trials

    file_exists = os.path.isfile(filename)
    
    # Define the data strictly using the averaged dictionaries
    row_data = {
        "Num Bits": num_bits,
        "Tolerance": tolerance,
        "Disjoint Bags": disjoint_bags,
        "Total Variants": total_variants,
        "Successful Trials": successful_trials,
        
        "Reg Setup Time (s)": round(avg_reg.get('setup_time', 0), 6),
        "Reg Derive Time (s)": round(avg_reg.get('derive_time', 0), 6),
        "Reg PRF Time (s)": round(avg_reg.get('total_prf_time', 0), 6),
        "Reg Enc Time (s)": round(avg_reg.get('total_enc_time', 0), 6),
        "Reg Total All Bags (s)": round(avg_reg.get('server_total_reg_time', 0), 6),
        
        "Auth Phase 1 Total (s)": round(avg_auth.get('init_total_auth_time', 0), 6),
        "Auth Global Face ZKP (s)": round(avg_auth.get('init_global_face_zkp_time', 0), 6),
        "Auth ZKP (s)": round(avg_auth.get('init_zkp_time', 0), 6),
        "Auth Server OT Prep (s)": round(avg_auth.get('init_server_ot_gen_time', 0), 6),
        "Auth Client OT Eval (s)": round(avg_auth.get('init_client_ot_eval_time', 0), 6),
        
        "Audit Phase 2 Total (s)": round(avg_auth.get('total_global_audit_time', 0), 6),
        "Audit Derive Time (s)": round(avg_auth.get('audit_derive_variants_time', 0), 6),
        "Audit PRF Time (s)": round(avg_auth.get('audit_prf_time', 0), 6),
        "Audit KDF & Dec (s)": round(avg_auth.get('audit_kdf_decrypt_time', 0), 6),
        
        "Total Auth+Audit Session (s)": round(avg_session, 6),
        "FP Rate": round(fp_rate, 6),
        "FN Rate": round(fn_rate, 6),
    }

    # Extract headers directly from the dictionary keys to prevent CSV mapping errors
    headers = list(row_data.keys())

    # Append to the file
    with open(filename, mode='a', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=headers)
        if not file_exists:
            writer.writeheader() 
        writer.writerow(row_data)

    print(f"✅ Averaged profiling data ({successful_trials}/{num_trials} successes) saved to {filename}")
    return True

## 6. False-Positive / False-Negative Analysis

In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor

def compute_fp_fn_rates(
    num_bits: int,
    tolerance: int,
    disjoint_bags: int,
    facial_data: dict,
    num_people: int = 100,
    num_pairs: int = 200,
    bag_trials: int = 5,
    k_bits: int = 12,
    n_workers: int = None,
) -> tuple:
    """
    Vectorised + multi-threaded FP/FN computation.

    Optimisations vs original:
      - Inner pair loop replaced by batched numpy matmul  (all pairs at once)
      - Outer people loop parallelised with ThreadPoolExecutor
        (numpy releases the GIL so threads run concurrently)

    Semantics identical to original: same sampling strategy, same auth logic.
    """
    if not facial_data:
        raise ValueError("facial_data is empty — load face embeddings first (Section 4).")
    if n_workers is None:
        n_workers = os.cpu_count() or 4

    person_ids = list(facial_data.keys())
    n = len(person_ids)

    # Pre-convert to float64 numpy arrays once (shared read-only across threads)
    faces_arr = [np.array(facial_data[pid], dtype=np.float64) for pid in person_ids]

    def _process_one(seed: int):
        rng = np.random.default_rng(seed)

        # ── Select genuine person ──────────────────────────────────────────────
        sel_idx   = int(rng.integers(n))
        sel_faces = faces_arr[sel_idx]          # (n_scans, D)
        template  = sel_faces.mean(axis=0)      # (D,)
        emb_dim   = template.shape[0]
        other_idx = [i for i in range(n) if i != sel_idx]

        trial_fp = 0.0
        trial_fn = 0.0

        for _ in range(bag_trials):
            # ── Fresh random bags ──────────────────────────────────────────────
            bags = [CosineLSH(num_bits, emb_dim) for _ in range(disjoint_bags)]

            # ── Per-bag: reliable-bit mask + template bit vector ───────────────
            reliable_masks = []  # (M,) bool
            tmpl_bits_list = []  # (M,) bool

            for bag in bags:
                bag.filter_by_hyperplane(template, tolerance, k_bits=k_bits)
                uncertain = set(int(i) for i in bag.filtered_indices)
                reliable_masks.append(
                    np.array([i not in uncertain for i in range(num_bits)]))
                H = bag.hyperplanes                    # (M, D)
                tmpl_bits_list.append((H @ template) >= 0)  # (M,) bool

            # ── Batch-sample impostor probes ───────────────────────────────────
            oi = rng.integers(len(other_idx), size=num_pairs)
            fp_probes = np.stack([
                faces_arr[other_idx[i]][
                    rng.choice(len(faces_arr[other_idx[i]]),
                               size=min(2, len(faces_arr[other_idx[i]])),
                               replace=False)
                ].mean(axis=0)
                for i in oi
            ])                                         # (P, D)

            # ── Batch-sample genuine probes ────────────────────────────────────
            fn_probes = np.stack([
                sel_faces[
                    rng.choice(len(sel_faces),
                               size=min(2, len(sel_faces)),
                               replace=False)
                ].mean(axis=0)
                for _ in range(num_pairs)
            ])                                         # (P, D)

            fp_any_zero = np.zeros(num_pairs, dtype=bool)
            fn_any_zero = np.zeros(num_pairs, dtype=bool)

            for bag_i, bag in enumerate(bags):
                H   = bag.hyperplanes                  # (M, D)
                rel = reliable_masks[bag_i]            # (M,) bool
                tb  = tmpl_bits_list[bag_i]            # (M,) bool

                # Batch hash all probes at once: (P, M) bool
                fp_bits = (fp_probes @ H.T) >= 0
                fn_bits = (fn_probes @ H.T) >= 0

                # Disagreements on reliable bits only → (P,) int count
                fp_dist = ((fp_bits ^ tb) & rel).sum(axis=1)
                fn_dist = ((fn_bits ^ tb) & rel).sum(axis=1)

                # Auth succeeds if ANY bag has distance == 0
                fp_any_zero |= (fp_dist == 0)
                fn_any_zero |= (fn_dist == 0)

            trial_fp += fp_any_zero.mean()
            trial_fn += (~fn_any_zero).mean()

        return trial_fp / bag_trials, trial_fn / bag_trials

    rng_master = np.random.default_rng()
    seeds = rng_master.integers(0, 2**31, size=num_people)

    with ThreadPoolExecutor(max_workers=n_workers) as pool:
        results = list(pool.map(_process_one, seeds))

    fp_vals, fn_vals = np.array(results).T
    return float(fp_vals.mean()), float(fn_vals.mean())

## 7. Parameter Sweep

Sweeps over `(M, t, d)` configurations to characterise the accuracy–security trade-off.
Requires `facial_data` to be loaded (Section 4). Results are appended to
`pake_profiling_resnet50_big.csv`.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ResNet50 (buffalo_l) Parameter Sweep
# Targets viable (M, t, d) configs from tablestar512 analysis.
# Results written to pake_profiling_resnet50.csv
# ═══════════════════════════════════════════════════════════════════════════════

import os, csv, time, random
import numpy as np

# ── Parameter grid (projected optimal configs: minimize FP→0, FN<10%) ──────────
BITS_LIST      = [24]
TOLERANCE_LIST = [10]
BAGS_LIST      = [20]

TIMING_TRIALS       = 1
MAX_VARIANTS_TIMING = 5_000_000
OUTPUT_FILE         = "pake_profiling_resnet50_big.csv"  # appends

HEADERS = [
    "Num Bits", "Tolerance", "Disjoint Bags", "Total Variants",
    "Session Total (s)", "FP Rate", "FN Rate", "Timing Trials",
    "Reg PRF Time (s)", "Reg Enc Time (s)", "Reg Total (s)",
    "Auth Total (s)", "Auth ZKP (s)", "Auth OT Prep (s)", "Auth OT Eval (s)",
    "Audit Total (s)", "Audit PRF Time (s)", "Audit KDF Dec (s)",
]

def adaptive_trials(total_variants, base_trials=TIMING_TRIALS, max_variants=MAX_VARIANTS_TIMING):
    fraction = 1.0 - min(total_variants / max_variants, 0.9)
    return 1

def _run_timing(num_bits, tolerance, disjoint_bags, num_trials):
    acc_reg, acc_auth, acc_session = {}, {}, 0.0
    successes = 0
    person_keys = list(facial_data.keys())
    for _ in range(num_trials):
        person   = random.choice(person_keys)
        raw_face = random.choice(facial_data[person])
        server = PAKE_server(disjoint_bags, number_of_hyperplanes=num_bits, tolerance=tolerance)
        ok, reg_metrics = server.register(raw_face)
        if not ok:
            continue
        client = PAKE_client(raw_face)
        t0 = time.time()
        ok  = server.authenticate(client)
        t1  = time.time()
        if not ok:
            continue
        for k, v in reg_metrics.items():
            acc_reg[k] = acc_reg.get(k, 0.0) + v
        for k, v in client.metrics.items():
            acc_auth[k] = acc_auth.get(k, 0.0) + v
        acc_session += (t1 - t0)
        successes   += 1
    if successes == 0:
        return None, None, None, 0
    avg_reg  = {k: v / successes for k, v in acc_reg.items()}
    avg_auth = {k: v / successes for k, v in acc_auth.items()}
    return avg_reg, avg_auth, acc_session / successes, successes

# ── Pre-flight ─────────────────────────────────────────────────────────────────
all_configs = [(m, t, b) for m in BITS_LIST for t in TOLERANCE_LIST for b in BAGS_LIST]
total = len(all_configs)
print(f"ResNet50 sweep: {total} configurations")
print(f"  M (bits)   : {BITS_LIST}")
print(f"  Tolerance  : {TOLERANCE_LIST}")
print(f"  Bags       : {BAGS_LIST}")
print(f"  Output     : {OUTPUT_FILE}\n")

file_exists = os.path.isfile(OUTPUT_FILE)
csv_file    = open(OUTPUT_FILE, mode="a", newline="")
writer      = csv.DictWriter(csv_file, fieldnames=HEADERS)
if not file_exists:
    writer.writeheader()
    csv_file.flush()

# ── Main loop ──────────────────────────────────────────────────────────────────
t_sweep_start = time.time()

for idx, (m, t, b) in enumerate(all_configs, 1):
    if t >= m:
        print(f"[{idx:>2}/{total}] M={m:>2}  t={t:>2}  d={b:>2}  SKIPPED (t >= M)")
        continue
    total_variants = b * (1 << t)
    n_trials       = adaptive_trials(total_variants)

    elapsed = time.time() - t_sweep_start
    eta_str = ""
    if idx > 1:
        avg_s     = elapsed / (idx - 1)
        remaining = avg_s * (total - idx + 1)
        eta_str   = f"  ETA {remaining/60:.0f} min"

    print(f"[{idx:>2}/{total}] M={m:>2}  t={t:>2}  d={b:>2}  "
          f"variants={total_variants:>7,}  trials={n_trials}{eta_str}")

    t_fpfn = time.time()
    fp, fn = compute_fp_fn_rates(num_bits=m, tolerance=t, disjoint_bags=b, facial_data=facial_data)
    print(f"       FP/FN:  FP={fp:.5f}  FN={fn:.5f}  ({time.time()-t_fpfn:.0f}s)")

    t_pake = time.time()
    avg_reg, avg_auth, avg_session, successes = _run_timing(m, t, b, n_trials)
    if avg_reg is None:
        avg_reg = avg_auth = {}
        avg_session = 0.0
    print(f"       Timing: session={avg_session:.3f}s  ({successes}/{n_trials} trials,  {time.time()-t_pake:.0f}s)")

    g = lambda d, k: round(d.get(k, 0), 6) if d else ""
    row = {
        "Num Bits":           m,   "Tolerance":       t,
        "Disjoint Bags":      b,   "Total Variants":  total_variants,
        "Session Total (s)":  round(avg_session, 6) if avg_session else "",
        "FP Rate":            round(fp, 6),
        "FN Rate":            round(fn, 6),
        "Timing Trials":      n_trials,
        "Reg PRF Time (s)":   g(avg_reg,  "total_prf_time"),
        "Reg Enc Time (s)":   g(avg_reg,  "total_enc_time"),
        "Reg Total (s)":      g(avg_reg,  "server_total_reg_time"),
        "Auth Total (s)":     g(avg_auth, "init_total_auth_time"),
        "Auth ZKP (s)":       g(avg_auth, "init_zkp_time"),
        "Auth OT Prep (s)":   g(avg_auth, "init_server_ot_gen_time"),
        "Auth OT Eval (s)":   g(avg_auth, "init_client_ot_eval_time"),
        "Audit Total (s)":    g(avg_auth, "total_global_audit_time"),
        "Audit PRF Time (s)": g(avg_auth, "audit_prf_time"),
        "Audit KDF Dec (s)":  g(avg_auth, "audit_kdf_decrypt_time"),
    }
    writer.writerow(row)
    csv_file.flush()

csv_file.close()
total_min = (time.time() - t_sweep_start) / 60
print(f"\n✅ Done — {total} configs in {total_min:.1f} min → {OUTPUT_FILE}")

# ── Summary ────────────────────────────────────────────────────────────────────
import csv as _csv
rows = list(_csv.DictReader(open(OUTPUT_FILE)))
candidates = [r for r in rows if r["FN Rate"] != "" and float(r["FN Rate"]) < 0.05]
candidates.sort(key=lambda r: float(r["FP Rate"]))
print(f"\nConfigs with FN<5%, sorted by FP Rate:")
print(f"{'M':>4} {'t':>4} {'d':>4} {'Variants':>10}  {'FP':>9}  {'FN':>9}  {'Session(s)':>11}")
print("-" * 65)
for r in candidates:
    sess = r["Session Total (s)"] or "—"
    print(f'  {r["Num Bits"]:>3} {r["Tolerance"]:>3} {r["Disjoint Bags"]:>3} '
          f'{r["Total Variants"]:>10}  {float(r["FP Rate"]):>9.5f}  '
          f'{float(r["FN Rate"]):>9.5f}  {sess:>11}')
